# Advancing RAG's Limitation: Multimodal RAG Architecture

## 0. Instructions & Context
The main part of the projectb code is contained in the python file `multimodal.py`, that can be run via terminal with the command `streamlit run multimodal.py`.

To run the streamlit applicaion smoothly and to not mess up with existing configurations locally, we set up an environment called `envmultimodal` via `conda`:


`(base) conda create -n "envmultimodal"`

`(base) conda activate envmultimodal`


`(envmultimodal) pip install streamlit chromadb sentence-transformer requests "unstructured[all-docs" fitz #or pyumpdf`
    
One of the main novelties lies within the choice of the Ollama models. Two models have been chosen, tinyllama and moondream. The new model moondream has been chosen fue to its image processing capabilities.


## 1. Project Motivation
This project originates from the RAG Architectures workshop held in early March. During the workshop, we explored several local RAG implementations (Naive RAG, DeepDoc RAG, TableBook RAG) and analyzed their common failure modes. 
From the workshops' results, DeepDoc emerged as the most reliable and complete architechture, yet with a crucial limitation: dealing with visualizations. 

In the context of an AI tutor for the Information Systems department, such limitation is crucial since most learning materials heavily rely on infographics, ranging from graphs to pipelines. This major limitation identified in all 3 approaches (Naive, DeepDoc, TableBook) is the **Image Blind Spot** (Failure Code 'I').

In typical RAG systems, charts, diagrams, and image-only slides produce empty chunks because the model used (in our case tinyllama) lacks vision capabilities. 

To solve this, the programming project aims at developing a **Multimodal RAG** deployed via a simialar Streamlit app as the one shpwed during the workshops/presentation. This architecture combines layout-aware text extraction with an active vision model to caption images before indexing, ensuring that visual diagrams become searchable text context.

## 2. System Architecture Setup
The system relies on local, privacy-preserving models running via Ollama:
*   **Text Generation Model:** `tinyllama` 
*   **Vision Model:** `moondream` 
*   **Vector Database:** `ChromaDB` using the `all-MiniLM-L6-v2` embedding model

In the code cell below we can see the configuration needed to run the streamlit application:

In [ ]:
import chromadb
from chromadb.utils import embedding_functions
import requests
import fitz 
import base64
import os
from unstructured.partition.auto import partition
from unstructured.documents.elements import Title

TEMP_DIR = "temp_files_multimodal"
CHROMA_DIR = "chroma_data_multimodal"
os.makedirs(TEMP_DIR, exist_ok=True)

OLLAMA_URL = "http://localhost:11434/api/generate"
TEXT_MODEL = "tinyllama"
VISION_MODEL = "moondream"
VISION_TIMEOUT = 600
TEXT_TIMEOUT = 120

With respect to the former approach, an important design choice has been made at this time and that is to pull 2 different models from Ollama. Tinyllama is assigned as the text model, whereas moondream as the vision model. 

Now that we have 2 different models in our architechture, 2 extraction strategies need to be instantiated. 

In [ ]:
#text extraction
def extract_text_chunks(file_path: str, file_name: str) -> list[dict]:
    chunks, current, current_type = [], [], None
    try:
        elements = partition(filename=file_path)
        for el in elements:
            text = str(el).strip()
            if not text:
                continue
            el_type = type(el).__name__
            if isinstance(el, Title) and current:
                chunks.append({"text": "\n".join(t for t, _ in current),
                                "type": current_type or "NarrativeText"})
                current = []
            current.append((text, el_type))
            current_type = el_type
        if current:
            chunks.append({"text": "\n".join(t for t, _ in current),
                            "type": current_type or "NarrativeText"})
    except Exception as e:
        st.warning(f"Text parsing warning for {file_name}: {e}")
    return chunks

Above we can see how text extraction is handled. The unstructured library uses the partition method in order to distinguish between the visual and structural elements of the input. 
As pointed out during our presentation, more sophisticated chunking strategies like these are more efficient and adopted than blind/fixed size chunking. 

Thw functiion above returns the chunks that are also linked with important semantic metadata stored in tags. Such strategy helps preserving th eoriginal context andf enables an intelligent retrieval in the vector database. 

In [ ]:
def caption_image(img_bytes: bytes) -> str | None:
    img_b64 = base64.b64encode(img_bytes).decode("utf-8")
    payload  = {
        "model":  VISION_MODEL,
        "prompt": (
            "Describe this diagram or image in detail. "
            "Focus on any data, text labels, and relationships shown. "
            "Be concise but complete."
        ),
        "images": [img_b64],
        "stream": False,
    }
    try:
        res = requests.post(OLLAMA_URL, json=payload, timeout=VISION_TIMEOUT)
        if res.status_code == 200:
            return res.json().get("response", "").strip()
    except requests.exceptions.Timeout:
        st.warning("⏱️ Vision model timed out on an image — skipping.")
    except Exception as e:
        st.warning(f"Vision error: {e}")
    return None


def extract_image_chunks(file_path: str, file_name: str,
                         progress_placeholder) -> list[dict]:
    chunks = []
    try:
        doc        = fitz.open(file_path)
        all_images = [
            (page_num, img)
            for page_num in range(len(doc))
            for img in doc.load_page(page_num).get_images(full=True)
        ]
        total = len(all_images)
        for i, (page_num, img) in enumerate(all_images):
            xref       = img[0]
            base_image = doc.extract_image(xref)
            if (base_image.get("width",  0) < MIN_IMG_WIDTH or
                    base_image.get("height", 0) < MIN_IMG_HEIGHT):
                continue

            progress_placeholder.text(
                f"Captioning image {i + 1}/{total} on page {page_num + 1}…"
            )
            caption = caption_image(base_image["image"])
            if caption:
                chunks.append({
                    "text": f"[IMAGE DESCRIPTION — Page {page_num + 1}]: {caption}",
                    "type": "ImageCaption",
                })
    except Exception as e:
        st.warning(f"Image extraction failed for {file_name}: {e}")
    return chunks

The above functions aim at solving the limitation explained at the beginning of the notebook, that is the failures classified under the Image Blind Spot category. 

**extract_image_chunks** function exploits the fitz library and inspect a document looking for images to embedd. When it locates an image, it extracts the bytes, updates and passes the image to the captioning function. If a caption is successfully generated, it packages the description with its original page number and assigns it the metadata tag "type": "ImageCaption" before adding it to the chunk list.

**caption_image** receives the bytes and encpdes them in base64 format. Then prompts to the local API by telling it to describ3e the visual it receives and to pay attention to menaingful information like data, labels and relationships. Local vision models like *moondream* can be computationtally expensive, that's why the timeout size has been increases with resèect to the timput for the text model *tinyllama*. 

These functions successfully convert opaque visual data into rich, searchable text strings. This ensures that the RAG pipeline can retrieve and answer questions based on visual evidence that implementations like Naive RAG or DeepDoc RAG would completely miss

## 3. The Knowledge Base: a summary of my master thesis project 
The knowledge base is represented by a document outlining my master thesis project which focuses on integrating XAI techniques in the medical setting, specifically in the diagnosis of pancreatic cancer. 

In the document we can find 8 images that will undergo the embedding explained before. Such visuals describe the project pipeline, as well outputs of the explainable AI techniques (visual explanations including SHAP beeswarm plots, waterfall charts, and LIME local charts). 
Some are presented below and we are going to hold on to these when it comes to interrogate our new RAG:

![Project Pipeline Architecture](data/fig1.png)
![SHAP and LIME Plots](data/fig2.png)
![Correctness](data/fig3.png)

The rest of the knowledge base can be consulted in the Summary.pdf file.

## 4. Streamlit App

Now that we delved deeper into the main novelties of the proposed multimodal RAG, let's see how it appears on the Streamlit implementation:
![Streamlit app](data/streamlit.png)

The baove snapshots shows how the current implementation already takes into consideration past ingested knowledge and this happens thanks to the presence of the vector database ChromaDB. 

## 5. Testing the Multimodal RAG
The presented RAG is tested against the following prompts:

1. What machine learning models and Explainable AI (XAI) techniques were implemented in the project?
2. Based on the pipeline diagram, what specific metrics are used in the Quantitative Evaluation of the XAI?
3. Can you describe the visual outputs shown in the SHAP waterfall chart or LIME local chart?
4. What are the proposed ways for clinical experts to collaborate on this research?
5. What specific chemotherapy drugs or surgical treatments are recommended for Pancreatic Ductal Adenocarcinoma in the thesis?

### 5.1 What machine learning models and Explainable AI (XAI) techniques were implemented in the project?
The RAG correctly identified 2 out of 3 machine learning models that will be benchmarked in the thesis, as well as both XAI approaches (SHAP and LIME). However, it was wrong in stating that logistic regression and this hallucination pattern will be present alongside the whole testing procedure, probably due to some limitations coming from the models' sizes and capabilities. 
![results1](data/fig4.png)


### 5.2 Based on the pipeline diagram, what specific metrics are used in the Quantitative Evaluation of the XAI?
The system almost identified the scope of the question, centering it around the evaluation but missed to just focus on the quantitative part. It guessed correctly most fo the qualitative crtieria. The quantitative metrics were not grasped, as we would have to read something like correctness, consistency and contrastivity. 
![results2](data/fig5.png)


### 5.3 Can you describe the visual outputs shown in the SHAP waterfall chart or LIME local chart?
The RAG successfully parsed the image, as we see some attempts to describing the layout of the graphs, but unfortunately all type of encountered data was either mis-intepreted or hallucinated. The RAG interreted the bar plot as a visualization representing the relationship between different types of cancers, whenin reality we talk about attribute contributions in PDAC across the target variable, being either healht, benign or cancer.
![results3](data/fig6.png)



### 5.4 What are the proposed ways for clinical experts to collaborate on this research?
Initially, the RAG based system gives out information not requested in the prompt, but in the second paragraph it suceeds in giving out a partially correct answer with respect to poetntial collaboration scenarios that canbe found in the knowledge base, although it remains quite vague. 
![results4](data/fig8.png)


### 5.5 What specific chemotherapy drugs or surgical treatments are recommended for Pancreatic Ductal Adenocarcinoma in the thesis?
The RAG based system doesn't invent anything outside the scope of the knowledge base, yet fails to properly inform the user that it doesn't know such information or by saying thet this question cannot be answered given the knowledge base provided, meaning that the question is out of scope. 
![results4](data/fig9.png)


## 6. Conclusions & Limitations

### 6.1 Bridging the visual gap
The updated RAG addresses what specified at the beginning of the notebook as one of the main limitations, that is the Image Blind Spot. This has been addressed via the integration of `PyMuPDF` for extraction purposes and `moondream` for the visual model responsible to caption the identified visualizations.

The proposed piepline trasforms the visualizations spotted in the knowledge base, medical diagrams for the sake of this example, into text strings and make such content visible for the system and to the vector database. 

In terms of retrieval, the proposed layout spots relevant chunks based on image captions, highlighting the importance of enriching our databases with etadata that can also categorize visual elements. 

Finally, when questioning the system with out of scope information (see Q5), it was successful in not hallucinating notn existent data, sticking to the provided materials, yet failed to explicitly answering with "I don't know this/I don't have this information". 

### 6.2 Limitations: the size of the models
The overall proposed architechture is sound but the testing reveald substantial limitations, primarily related to the chosen small models (Tinyllama and moondream).
The selected models were chosen with configurations applicable to the hardware available (Macbook Air - 8gb RAM and limited disk space). Moondream recognized some visuals like charts, yet failed to correctly inteprete the data and the relationships. The description of the graphics were too high level and therefore failed to correctly answer the test questions. 

Another difficulty, emerged in terms of negative constraints. The systems was instructed with a prompt specifying to state something like "I don't have this information", but instead it generated vague and out of scope responses. This indicates a limitation in the base model's ability to follow complex system prompts regarding boundaries.

### 6.3 Future Implementations
The proposed architechture can be considered as a sort of prototype/baseline for future works. Future products should undergo re-engineering in terms of LLM size, OCR Preprocessing and prompt engineering. 
In terms of model choice, while moondream is a powerful one, moving to a 7b or 13b parameter model (Llama3, Mistral) will for sure improve the model's reasoning ability. 

Moreover, some preprocessing condiering OCR should be combined with the chosen vision model to enrich the metadata and therefore refine the chunking and retrieval stages; in this way text labels inside charts could improve the cptioning of visual elements. 
Finally, refining the instructions could limit hallucinations and out of scope responses.

